# Machine Learning Practical: K-Means Clustering

**Dataset:** `diabetes_prediction_dataset.csv`

K-Means is an unsupervised learning algorithm. It groups records into 2 clusters here. The `diabetes` target is not used to create clusters.
Keep the CSV file in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("diabetes_prediction_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nFirst five rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

## Prepare Data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Do not use the diabetes label as an input feature.
X = df.drop("diabetes", axis=1)

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_columns),
    ("cat", categorical_transformer, categorical_columns)
])

X_processed = preprocessor.fit_transform(X)
print("Processed data shape:", X_processed.shape)

## Apply K-Means Algorithm

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_processed)

print("First 20 cluster labels:", clusters[:20])
print("Cluster counts:")
print(pd.Series(clusters).value_counts().sort_index())
print("Inertia:", kmeans.inertia_)
print("Silhouette Score:", silhouette_score(X_processed, clusters))

## Visualize Clusters

In [ ]:
from sklearn.decomposition import PCA

# PCA reduces the processed features to 2 dimensions for plotting.
X_2d = PCA(n_components=2).fit_transform(X_processed.toarray()
    if hasattr(X_processed, "toarray") else X_processed)

plt.figure(figsize=(8, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, alpha=0.5)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters")
plt.show()